### Predicción de partidos de futbol

En trabajos anteriores siempre hemos utilizado un promedio de las estadísticas principales que usamos para la predicción de partidos (tiros, goles, xG, etc..), sin embargo, a pesar de que ha funcionado, es mejorable, ya que la dinámica cambia considerablemente según el calendario, rivales, lesiones entre otras cosas. 

La propuesta en este caso es el $suavizado$ $exponencial$ (EWMA), corrige la anterior propuesta dándole un "decaimiento"  a la historia.


$EWMA_t$ $=$ $\alpha x_t + (1-\alpha) EWMA_{t-1}$

Si $\alpha$ es alto, el modelo reacciona rápido a cambios de racha, si es bajo, el modelo tiene más memoria a largo plazo y es menos sensible al ruido de un mal partido.

Lo primero a realizar es cargar las bases de datos de Footbal-Data y de Understat para obtener todas las estadísticas que vamos a utilizar.

In [1]:
import pandas as pd
import numpy as np
import requests
import io
import joblib
import os
import re
import json
from scipy.stats import poisson
import xgboost as xgb
from tensorflow.keras.models import load_model
import warnings
warnings.filterwarnings('ignore')

print("⚙️ INICIANDO MEGA-PROTOCOLO DE VIERNES: IA + POISSON + KELLY...")

#Descargar datos de Footbal Data
print("\n🌐 1. Descargando resultados y cuotas actualizadas...")
ligas = ['E0', 'SP1', 'I1', 'D1', 'F1']
temporadas = ['2122', '2223', '2324', '2425', '2526']
base_url = "https://www.football-data.co.uk/mmz4281/{}/{}.csv"

dfs = []
for liga in ligas:
    for temp in temporadas:
        try:
            url = base_url.format(temp, liga)
            s = requests.get(url).content
            df_temp = pd.read_csv(io.StringIO(s.decode('latin-1')))
            df_temp['League_ID'] = liga
            dfs.append(df_temp)
        except:
            continue

df_main = pd.concat(dfs, ignore_index=True)
print(f"   ✅ Base clásica descargada: {len(df_main)} partidos.")

# Limpieza y Promedio de Cuotas
cols_home = [c for c in df_main.columns if c.endswith('H') and c not in ['FTHG', 'HTHG']]
cols_draw = [c for c in df_main.columns if c.endswith('D')]
cols_away = [c for c in df_main.columns if c.endswith('A') and c not in ['FTAG', 'HTAG']]

for col in cols_home + cols_draw + cols_away:
    df_main[col] = pd.to_numeric(df_main[col], errors='coerce')

df_main['Odds_Home'] = df_main[cols_home].mean(axis=1)
df_main['Odds_Draw'] = df_main[cols_draw].mean(axis=1)
df_main['Odds_Away'] = df_main[cols_away].mean(axis=1)

df_clean = df_main.dropna(subset=['Odds_Home', 'Odds_Away', 'FTHG']).copy()
df_clean['Date'] = pd.to_datetime(df_clean['Date'], dayfirst=True)

df_clean['Home_Pts'] = np.select(
    [(df_clean['FTHG'] > df_clean['FTAG']), (df_clean['FTHG'] == df_clean['FTAG']), (df_clean['FTHG'] < df_clean['FTAG'])], 
    [3, 1, 0]
)
df_clean['Away_Pts'] = np.select(
    [(df_clean['FTHG'] > df_clean['FTAG']), (df_clean['FTHG'] == df_clean['FTAG']), (df_clean['FTHG'] < df_clean['FTAG'])], 
    [0, 1, 3]
)

# Sistema ELO y rachas básicas 
print("\n🧮 2. Calculando el sistema ELO y Rachas Básicas...")
def agregar_elo(df):
    df = df.sort_values('Date').copy()
    elo_inicial = 1500; k_factor = 20; ventaja_local = 50
    elo_dict = {}
    elo_home_h, elo_away_h = [], []

    for idx, row in df.iterrows():
        home, away = str(row['HomeTeam']).strip(), str(row['AwayTeam']).strip()
        elo_h, elo_a = elo_dict.get(home, elo_inicial), elo_dict.get(away, elo_inicial)
        elo_home_h.append(elo_h); elo_away_h.append(elo_a)

        goles_h, goles_a = row['FTHG'], row['FTAG']
        if goles_h > goles_a: W_h = 1.0
        elif goles_h == goles_a: W_h = 0.5
        else: W_h = 0.0

        dr = (elo_h + ventaja_local) - elo_a
        We_h = 1 / (10 ** (-dr / 400) + 1)
        diff = abs(goles_h - goles_a)
        multiplicador = 1.0 if diff < 2 else np.sqrt(diff)

        cambio = k_factor * multiplicador * (W_h - We_h)
        elo_dict[home] = elo_h + cambio
        elo_dict[away] = elo_a - cambio

    df['Elo_Home'] = elo_home_h
    df['Elo_Away'] = elo_away_h
    df['Elo_Diff'] = (df['Elo_Home'] + ventaja_local) - df['Elo_Away']
    return df

df_elo = agregar_elo(df_clean)

# Calcular Rachas Básicas
def calcular_rachas_complejas(df):
    home = df[['Date', 'HomeTeam', 'HS', 'AS', 'HST', 'AST', 'FTHG', 'FTAG', 'Home_Pts']].rename(
        columns={'HomeTeam': 'Team', 'HS': 'Shots_Total_For', 'AS': 'Shots_Total_Ag', 'HST': 'Shots_Target_For', 'AST': 'Shots_Target_Ag', 'FTHG': 'Goals_For', 'FTAG': 'Goals_Ag', 'Home_Pts': 'Pts'})
    away = df[['Date', 'AwayTeam', 'AS', 'HS', 'AST', 'HST', 'FTAG', 'FTHG', 'Away_Pts']].rename(
        columns={'AwayTeam': 'Team', 'AS': 'Shots_Total_For', 'HS': 'Shots_Total_Ag', 'AST': 'Shots_Target_For', 'HST': 'Shots_Target_Ag', 'FTAG': 'Goals_For', 'FTHG': 'Goals_Ag', 'Away_Pts': 'Pts'})
    combined = pd.concat([home, away]).sort_values(['Team', 'Date'])
    metricas = ['Shots_Total_For', 'Shots_Total_Ag', 'Shots_Target_For', 'Shots_Target_Ag', 'Goals_For', 'Goals_Ag', 'Pts']
    for m in metricas:
        combined[f'Form_{m}'] = combined.groupby('Team')[m].transform(lambda x: x.shift().rolling(window=5, min_periods=3).mean())
    return combined

df_rachas = calcular_rachas_complejas(df_elo)
df_rachas['Team'] = df_rachas['Team'].astype(str).str.strip()

cols_metricas = [c for c in df_rachas.columns if c.startswith('Form_')]
cols_rachas_clean = ['Date', 'Team'] + cols_metricas

df_final_elo = pd.merge(df_elo, df_rachas[cols_rachas_clean], left_on=['Date', 'HomeTeam'], right_on=['Date', 'Team'], how='left')
df_final_elo.rename(columns={c: f"Home_{c.replace('Form_', '')}_Form" for c in cols_metricas}, inplace=True)
df_final_elo.drop(columns=['Team'], inplace=True)

df_final_elo = pd.merge(df_final_elo, df_rachas[cols_rachas_clean], left_on=['Date', 'AwayTeam'], right_on=['Date', 'Team'], how='left')
df_final_elo.rename(columns={c: f"Away_{c.replace('Form_', '')}_Form" for c in cols_metricas}, inplace=True)
df_final_elo.drop(columns=['Team'], inplace=True)
df_final_elo.fillna(0, inplace=True)

⚙️ INICIANDO MEGA-PROTOCOLO DE VIERNES: IA + POISSON + KELLY...

🌐 1. Descargando resultados y cuotas actualizadas...
   ✅ Base clásica descargada: 8540 partidos.

🧮 2. Calculando el sistema ELO y Rachas Básicas...


In [2]:
import pandas as pd
import json
import glob
import os

print("📂 LEYENDO ARCHIVOS JSON LOCALES DE UNDERSTAT...")

# Buscar todos los archivos que terminen en .json en la carpeta actual
archivos_json = glob.glob('*.json')
datos_partidos = []

for archivo in archivos_json:
    try:
        # Abrimos cada archivo 
        with open(archivo, 'r', encoding='utf-8') as f:
            partidos = json.load(f)
            
            # Extraemos la liga y la temporada del nombre del archivo (Ej: EPL_2025.json)
            nombre_base = archivo.replace('.json', '')
            partes = nombre_base.split('_')
            temp = partes[-1] # El año siempre será el último (2025)
            liga = "_".join(partes[:-1]) # El resto es la liga (EPL, La_liga)

            # Filtramos y organizamos los datos
            for p in partidos:
                if p['isResult']: # Solo partidos que ya terminaron
                    datos_partidos.append({
                        'Date': p['datetime'].split(' ')[0], 
                        'League': liga,
                        'Season': f"{temp}/{int(temp)+1}",
                        'HomeTeam_Und': p['h']['title'],
                        'AwayTeam_Und': p['a']['title'],
                        'Goals_Home': int(p['goals']['h']),
                        'Goals_Away': int(p['goals']['a']),
                        'xG_Home': float(p['xG']['h']),      
                        'xG_Away': float(p['xG']['a']),      
                        'Forecast_Win': float(p['forecast']['w']),   
                        'Forecast_Draw': float(p['forecast']['d']),  
                        'Forecast_Loss': float(p['forecast']['l'])   
                    })
        print(f"   ✅ Archivo {archivo} procesado.")
    except Exception as e:
        print(f"   ❌ Error al leer {archivo}: {e}")

if len(datos_partidos) > 0:
    # Construimos el DataFrame
    df_xg = pd.DataFrame(datos_partidos)
    df_xg['Date'] = pd.to_datetime(df_xg['Date'])
    df_xg = df_xg.sort_values('Date').reset_index(drop=True)

    print(f"\n🚀 ¡ÉXITO! Se armó la tabla de Expected Goals con {len(df_xg)} partidos.")
    display(df_xg.tail())
else:
    print("\n❌ No se encontraron archivos .json válidos en la carpeta.")

📂 LEYENDO ARCHIVOS JSON LOCALES DE UNDERSTAT...
   ✅ Archivo Bundesliga_2021.json procesado.
   ✅ Archivo Bundesliga_2022.json procesado.
   ✅ Archivo Bundesliga_2023.json procesado.
   ✅ Archivo Bundesliga_2024.json procesado.
   ✅ Archivo Bundesliga_2025.json procesado.
   ✅ Archivo EPL_2021.json procesado.
   ✅ Archivo EPL_2022.json procesado.
   ✅ Archivo EPL_2023.json procesado.
   ✅ Archivo EPL_2024.json procesado.
   ✅ Archivo EPL_2025.json procesado.
   ❌ Error al leer EWMA_resultados_xgb.json: string indices must be integers
   ✅ Archivo LaLiga_2021.json procesado.
   ✅ Archivo LaLiga_2022.json procesado.
   ✅ Archivo LaLiga_2023.json procesado.
   ✅ Archivo LaLiga_2024.json procesado.
   ✅ Archivo LaLiga_2025.json procesado.
   ✅ Archivo Ligue1_2021.json procesado.
   ✅ Archivo Ligue1_2022.json procesado.
   ✅ Archivo Ligue1_2023.json procesado.
   ✅ Archivo Ligue1_2024.json procesado.
   ✅ Archivo Ligue1_2025.json procesado.
   ❌ Error al leer oraculoxG_xgb.json: string indi

,Date,League,Season,HomeTeam_Und,AwayTeam_Und,Goals_Home,Goals_Away,xG_Home,xG_Away,Forecast_Win,Forecast_Draw,Forecast_Loss
8535,2026-03-22,Bundesliga,2025/2026,Augsburg,VfB Stuttgart,2,5,1.59097,3.87094,0.0474,0.0964,0.8562
8536,2026-03-22,Bundesliga,2025/2026,St. Pauli,Freiburg,1,2,1.39653,2.74421,0.1151,0.1774,0.7075
8537,2026-03-22,Bundesliga,2025/2026,Mainz 05,Eintracht Frankfurt,2,1,1.38387,1.71748,0.2795,0.2810,0.4395
8538,2026-03-22,LaLiga,2025/2026,Real Madrid,Atletico Madrid,3,2,2.96605,1.40234,0.7734,0.1541,0.0725
8539,2026-03-22,SerieA,2025/2026,Fiorentina,Inter,1,1,1.81429,1.40088,0.4731,0.2716,0.2553


In [3]:
# Extraemos los datos en crudo 
# Buscamos todas las columnas que tengan la palabra 'Form' (son las rachas que calculamos)
columnas_viejas = [col for col in df_final_elo.columns if 'Form' in col]
df_elo_limpio = df_final_elo.drop(columns=columnas_viejas) #las eliminamos 

# Diccionario para juntas las bases de datos
diccionario_nombres = {
    'AC Milan': 'Milan', 'Arminia Bielefeld': 'Bielefeld', 'Athletic Club': 'Ath Bilbao',
    'Atletico Madrid': 'Ath Madrid', 'Bayer Leverkusen': 'Leverkusen', 'Borussia Dortmund': 'Dortmund',
    'Borussia M.Gladbach': "M'gladbach", 'Celta Vigo': 'Celta', 'Clermont Foot': 'Clermont',
    'Eintracht Frankfurt': 'Ein Frankfurt', 'Espanyol': 'Espanol', 'FC Cologne': 'FC Koln',
    'FC Heidenheim': 'Heidenheim', 'Greuther Fuerth': 'Greuther Furth', 'Hamburger SV': 'Hamburg',
    'Hertha Berlin': 'Hertha', 'Mainz 05': 'Mainz', 'Manchester City': 'Man City',
    'Manchester United': 'Man United', 'Newcastle United': 'Newcastle', 'Nottingham Forest': "Nott'm Forest",
    'Paris Saint Germain': 'Paris SG', 'RasenBallsport Leipzig': 'RB Leipzig', 'Rayo Vallecano': 'Vallecano',
    'Real Betis': 'Betis', 'Real Oviedo': 'Oviedo', 'Real Sociedad': 'Sociedad',
    'Real Valladolid': 'Valladolid', 'Saint-Etienne': 'St Etienne', 'St. Pauli': 'St Pauli',
    'VfB Stuttgart': 'Stuttgart', 'Parma Calcio 1913': 'Parma', 'Wolverhampton Wanderers': 'Wolves'
}

df_xg_clean = df_xg.copy()
df_xg_clean['HomeTeam_Und'] = df_xg_clean['HomeTeam_Und'].replace(diccionario_nombres)
df_xg_clean['AwayTeam_Und'] = df_xg_clean['AwayTeam_Und'].replace(diccionario_nombres)

# Fusión de bases 
cols_a_pegar = ['Date', 'HomeTeam_Und', 'AwayTeam_Und', 'xG_Home', 'xG_Away']

df_crudo = pd.merge(
    df_elo_limpio, 
    df_xg_clean[cols_a_pegar], 
    left_on=['Date', 'HomeTeam', 'AwayTeam'], 
    right_on=['Date', 'HomeTeam_Und', 'AwayTeam_Und'], 
    how='inner'
).drop(columns=['HomeTeam_Und', 'AwayTeam_Und'])

print(f"✅ Base 'df_crudo' lista: {len(df_crudo)} partidos. Pura, sin rachas y lista para Optuna.")

✅ Base 'df_crudo' lista: 8525 partidos. Pura, sin rachas y lista para Optuna.


In [4]:
df_crudo.tail(40)

,ï»¿Div,Date,Time,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,...,Odds_Home,Odds_Draw,Odds_Away,Home_Pts,Away_Pts,Elo_Home,Elo_Away,Elo_Diff,xG_Home,xG_Away
8485,SP1,2026-03-21,20:00,Sevilla,Valencia,0,2,A,0.0,2.0,...,2.127917,3.176250,2.961250,0,3,1453.996582,1499.098219,4.898363,0.505411,1.713300
8486,SP1,2026-03-21,15:15,Espanol,Getafe,1,2,A,0.0,2.0,...,2.322083,2.902500,2.827917,0,3,1465.441798,1487.285684,28.156114,1.512080,2.063840
8487,SP1,2026-03-21,13:00,Elche,Mallorca,2,1,H,0.0,0.0,...,2.003462,3.387222,3.115000,3,0,1413.164006,1460.567054,2.596953,1.516050,1.519390
8488,I1,2026-03-21,17:00,Milan,Torino,3,2,H,1.0,1.0,...,1.519231,4.877778,6.721923,3,0,1706.704183,1483.598963,273.105220,1.443370,2.372110
8489,D1,2026-03-21,14:30,Wolfsburg,Werder Bremen,0,1,A,0.0,0.0,...,2.227143,3.594500,2.572857,0,3,1435.279261,1473.958588,11.320673,1.056860,0.414924
8490,D1,2026-03-21,14:30,Heidenheim,Leverkusen,3,3,D,0.0,2.0,...,4.317143,4.511500,1.658214,1,1,1347.699705,1691.185659,-293.485954,1.737380,1.962870
8491,D1,2026-03-21,14:30,FC Koln,M'gladbach,3,3,D,2.0,2.0,...,2.216429,3.440000,2.692500,1,1,1424.181172,1478.195082,-4.013910,1.292930,1.081530
8492,E0,2026-03-21,20:00,Leeds,Brentford,0,0,D,0.0,0.0,...,2.270000,3.470625,2.585833,1,1,1458.151209,1577.491312,-69.340103,0.417296,0.302221
8493,E0,2026-03-21,17:30,Everton,Chelsea,3,0,H,1.0,0.0,...,2.905769,3.515556,2.091538,3,0,1552.591230,1644.344600,-41.753370,1.445320,1.351430
8494,E0,2026-03-21,15:00,Fulham,Burnley,3,1,H,0.0,0.0,...,1.654231,4.248889,4.893462,3,0,1538.190731,1361.431751,226.758979,3.484380,1.925560


Ahora, probaremos optuna para que encuentre la mejor combinación de hiperparámetros que mejoren el rendimiento ROI, entre los parámetros a encontrar se encuentra $\alpha$.

In [10]:
import optuna
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler
from scipy.stats import poisson
import warnings
warnings.filterwarnings('ignore')

print("🏛️ INICIANDO LABORATORIO CUANTITATIVO: 3-WAY SPLIT (SIN TRAMPAS)...")

# 1. Ordenamos cronológicamente
df_base = df_crudo.copy()
df_base['Date'] = pd.to_datetime(df_base['Date'])
df_base = df_base.sort_values('Date').reset_index(drop=True)

# Tamaños de los conjuntos
N = len(df_base)
train_idx = int(N * 0.70)
val_idx = int(N * 0.85)
# Train: 0 -> train_idx | Validation: train_idx -> val_idx | Test: val_idx -> N

def objetivo_honesto(trial): # intervalos de los hiperparámetros
    alpha_opt = trial.suggest_float('alpha', 0.05, 0.40)
    xgb_n_est = trial.suggest_int('xgb_n_est', 100, 300, step=50)
    xgb_lr = trial.suggest_float('xgb_lr', 0.01, 0.05)
    nn_lr = trial.suggest_float('nn_lr', 0.001, 0.01, log=True)
    nn_dropout = trial.suggest_float('nn_dropout', 0.2, 0.5)
    umbral_ev = trial.suggest_float('umbral_ev', 0.04, 0.10)
    
    # EMWA
    df_temp = df_base.copy()
    
    home = df_temp[['Date', 'HomeTeam', 'HS', 'AS', 'HST', 'AST', 'FTHG', 'FTAG', 'Home_Pts', 'xG_Home', 'xG_Away']].rename(
        columns={'HomeTeam': 'Team', 'HS': 'Shots_For', 'AS': 'Shots_Ag', 'HST': 'ST_For', 'AST': 'ST_Ag', 
                 'FTHG': 'G_For', 'FTAG': 'G_Ag', 'Home_Pts': 'Pts', 'xG_Home': 'xG_For', 'xG_Away': 'xG_Ag'})
    away = df_temp[['Date', 'AwayTeam', 'AS', 'HS', 'AST', 'HST', 'FTAG', 'FTHG', 'Away_Pts', 'xG_Away', 'xG_Home']].rename(
        columns={'AwayTeam': 'Team', 'AS': 'Shots_For', 'HS': 'Shots_Ag', 'AST': 'ST_For', 'HST': 'ST_Ag', 
                 'FTAG': 'G_For', 'FTHG': 'G_Ag', 'Away_Pts': 'Pts', 'xG_Away': 'xG_For', 'xG_Home': 'xG_Ag'})
    
    combined = pd.concat([home, away]).sort_values(['Team', 'Date'])
    
    metricas = ['Shots_For', 'Shots_Ag', 'ST_For', 'ST_Ag', 'G_For', 'G_Ag', 'Pts', 'xG_For', 'xG_Ag']
    for m in metricas:
        combined[f'Form_{m}'] = combined.groupby('Team')[m].transform(lambda x: x.shift().ewm(alpha=alpha_opt, adjust=False).mean())
        
    cols_form = ['Date', 'Team'] + [f'Form_{m}' for m in metricas]
    df_temp = pd.merge(df_temp, combined[cols_form], left_on=['Date', 'HomeTeam'], right_on=['Date', 'Team'], how='left').rename(columns={f'Form_{m}': f'EWMA_Home_{m}' for m in metricas}).drop(columns=['Team'])
    df_temp = pd.merge(df_temp, combined[cols_form], left_on=['Date', 'AwayTeam'], right_on=['Date', 'Team'], how='left').rename(columns={f'Form_{m}': f'EWMA_Away_{m}' for m in metricas}).drop(columns=['Team'])
    
    df_temp.fillna(1.0, inplace=True)
    
    df_temp['Proj_xG_Home'] = (df_temp['EWMA_Home_xG_For'] + df_temp['EWMA_Away_xG_Ag']) / 2.0
    df_temp['Proj_xG_Away'] = (df_temp['EWMA_Away_xG_For'] + df_temp['EWMA_Home_xG_Ag']) / 2.0
    # matriz de poisson
    def calc_poisson(row):
        try:
            p_H = [poisson.pmf(k, max(0.1, row['Proj_xG_Home'])) for k in range(6)]
            p_A = [poisson.pmf(k, max(0.1, row['Proj_xG_Away'])) for k in range(6)]
            matriz = np.outer(p_H, p_A)
            return pd.Series([np.tril(matriz, -1).sum(), np.trace(matriz), np.triu(matriz, 1).sum(), 1 - (matriz[0:3, 0:3].sum())])
        except: return pd.Series([0.33, 0.33, 0.34, 0.5])

    df_temp[['Poisson_1', 'Poisson_X', 'Poisson_2', 'Poisson_O25']] = df_temp.apply(calc_poisson, axis=1)
    
    features = [
        'Elo_Diff', 'Elo_Home', 'Elo_Away', 'Odds_Home', 'Odds_Draw', 'Odds_Away',
        'EWMA_Home_Shots_For', 'EWMA_Home_ST_For', 'EWMA_Home_G_For', 'EWMA_Home_G_Ag', 'EWMA_Home_Pts',
        'EWMA_Away_Shots_For', 'EWMA_Away_ST_For', 'EWMA_Away_G_For', 'EWMA_Away_G_Ag', 'EWMA_Away_Pts',
        'Proj_xG_Home', 'Proj_xG_Away', 'Poisson_1', 'Poisson_X', 'Poisson_2', 'Poisson_O25'
    ]
    df_temp['Target'] = np.select([(df_temp['FTHG'] > df_temp['FTAG']), (df_temp['FTHG'] == df_temp['FTAG']), (df_temp['FTHG'] < df_temp['FTAG'])], [0, 1, 2])
    
    # separación de test, train y val
    X = df_temp[features]
    y = df_temp['Target']
    
    X_train, y_train = X.iloc[:train_idx], y.iloc[:train_idx]
    X_val, y_val = X.iloc[train_idx:val_idx], y.iloc[train_idx:val_idx]
    df_val_real = df_temp.iloc[train_idx:val_idx].reset_index(drop=True) # Para checar cuotas
    
    scaler = StandardScaler()
    X_train_sc = scaler.fit_transform(X_train)
    X_val_sc = scaler.transform(X_val)
    
    # entrenamiento
    xgb_model = XGBClassifier(n_estimators=xgb_n_est, learning_rate=xgb_lr, max_depth=3, random_state=42, n_jobs=-1)
    xgb_model.fit(X_train.values, y_train.values)
    p_xgb = xgb_model.predict_proba(X_val.values)
    
    tf.get_logger().setLevel('ERROR')
    nn_model = Sequential([
        Input(shape=(X_train.shape[1],)),
        Dense(32, activation='relu'), BatchNormalization(), Dropout(nn_dropout),
        Dense(16, activation='relu'), BatchNormalization(), Dropout(nn_dropout),
        Dense(3, activation='softmax')
    ])
    nn_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=nn_lr), loss='sparse_categorical_crossentropy')
    es = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True, verbose=0)
    nn_model.fit(X_train_sc, y_train.values, validation_split=0.2, epochs=25, batch_size=32, callbacks=[es], verbose=0)
    
    p_nn = nn_model.predict(X_val_sc, verbose=0)
    p_ens = (p_xgb + p_nn) / 2.0
    
    # evaluación en validación 
    saldo = 0.0
    apostado = 0
    
    for i in range(len(df_val_real)):
        row = df_val_real.iloc[i]
        odds = [row['Odds_Home'], row['Odds_Draw'], row['Odds_Away']]
        probs = p_ens[i]
        real_target = row['Target']
        
        odd_1X = (odds[0] * odds[1]) / (odds[0] + odds[1])
        odd_X2 = (odds[1] * odds[2]) / (odds[1] + odds[2])
        ev_1 = (probs[0] * odds[0]) - 1; ev_2 = (probs[2] * odds[2]) - 1
        ev_1X = ((probs[0] + probs[1]) * odd_1X) - 1; ev_X2 = ((probs[1] + probs[2]) * odd_X2) - 1
        
        bet = None
        if ev_1 > umbral_ev and probs[0] > 0.45: bet = {'res': [0], 'odd': odds[0]}
        elif ev_1X > umbral_ev: bet = {'res': [0, 1], 'odd': odd_1X}
        elif ev_2 > umbral_ev and probs[2] > 0.45: bet = {'res': [2], 'odd': odds[2]}
        elif ev_X2 > umbral_ev: bet = {'res': [1, 2], 'odd': odd_X2}
        
        if bet:
            apostado += 10
            if real_target in bet['res']: saldo += (10 * bet['odd']) - 10
            else: saldo -= 10
            
    if apostado < 100: return -100.0 # Castigo por no apostar
    return (saldo / apostado) * 100

optuna.logging.set_verbosity(optuna.logging.WARNING)
study_honesto = optuna.create_study(direction='maximize')
study_honesto.optimize(objetivo_honesto, n_trials=15)

print("\n🏆 ¡OPTUNA TERMINÓ LA BÚSQUEDA!")
best_params = study_honesto.best_params
print(f"💰 MEJOR ROI en Validación: {study_honesto.best_value:.2f}%")
print(f"👉 Alfa EWMA Elegido: {best_params['alpha']:.4f}")

🏛️ INICIANDO LABORATORIO CUANTITATIVO: 3-WAY SPLIT (SIN TRAMPAS)...

🏆 ¡OPTUNA TERMINÓ LA BÚSQUEDA!
💰 MEJOR ROI en Validación: 16.65%
👉 Alfa EWMA Elegido: 0.3198


In [14]:
# Usamos la misma lógica para crear el DF final pero con el Alfa ganador
df_final = df_base.copy()
home = df_final[['Date', 'HomeTeam', 'HS', 'AS', 'HST', 'AST', 'FTHG', 'FTAG', 'Home_Pts', 'xG_Home', 'xG_Away']].rename(columns={'HomeTeam': 'Team', 'HS': 'Shots_For', 'AS': 'Shots_Ag', 'HST': 'ST_For', 'AST': 'ST_Ag', 'FTHG': 'G_For', 'FTAG': 'G_Ag', 'Home_Pts': 'Pts', 'xG_Home': 'xG_For', 'xG_Away': 'xG_Ag'})
away = df_final[['Date', 'AwayTeam', 'AS', 'HS', 'AST', 'HST', 'FTAG', 'FTHG', 'Away_Pts', 'xG_Away', 'xG_Home']].rename(columns={'AwayTeam': 'Team', 'AS': 'Shots_For', 'HS': 'Shots_Ag', 'AST': 'ST_For', 'HST': 'ST_Ag', 'FTAG': 'G_For', 'FTHG': 'G_Ag', 'Away_Pts': 'Pts', 'xG_Away': 'xG_For', 'xG_Home': 'xG_Ag'})
combined = pd.concat([home, away]).sort_values(['Team', 'Date'])

metricas = ['Shots_For', 'Shots_Ag', 'ST_For', 'ST_Ag', 'G_For', 'G_Ag', 'Pts', 'xG_For', 'xG_Ag']
# seleccionamos el mejor alpha
for m in metricas: combined[f'Form_{m}'] = combined.groupby('Team')[m].transform(lambda x: x.shift().ewm(alpha=best_params['alpha'], adjust=False).mean())
cols_form = ['Date', 'Team'] + [f'Form_{m}' for m in metricas]

df_final = pd.merge(df_final, combined[cols_form], left_on=['Date', 'HomeTeam'], right_on=['Date', 'Team'], how='left').rename(columns={f'Form_{m}': f'EWMA_Home_{m}' for m in metricas}).drop(columns=['Team'])
df_final = pd.merge(df_final, combined[cols_form], left_on=['Date', 'AwayTeam'], right_on=['Date', 'Team'], how='left').rename(columns={f'Form_{m}': f'EWMA_Away_{m}' for m in metricas}).drop(columns=['Team'])
df_final.fillna(1.0, inplace=True)

df_final['Proj_xG_Home'] = (df_final['EWMA_Home_xG_For'] + df_final['EWMA_Away_xG_Ag']) / 2.0
df_final['Proj_xG_Away'] = (df_final['EWMA_Away_xG_For'] + df_final['EWMA_Home_xG_Ag']) / 2.0

def calc_poisson_final(row):
    try:
        p_H = [poisson.pmf(k, max(0.1, row['Proj_xG_Home'])) for k in range(6)]
        p_A = [poisson.pmf(k, max(0.1, row['Proj_xG_Away'])) for k in range(6)]
        matriz = np.outer(p_H, p_A)
        return pd.Series([np.tril(matriz, -1).sum(), np.trace(matriz), np.triu(matriz, 1).sum(), 1 - (matriz[0:3, 0:3].sum())])
    except: return pd.Series([0.33, 0.33, 0.34, 0.5])

df_final[['Poisson_1', 'Poisson_X', 'Poisson_2', 'Poisson_O25']] = df_final.apply(calc_poisson_final, axis=1)

features = [
    'Elo_Diff', 'Elo_Home', 'Elo_Away', 'Odds_Home', 'Odds_Draw', 'Odds_Away',
    'EWMA_Home_Shots_For', 'EWMA_Home_ST_For', 'EWMA_Home_G_For', 'EWMA_Home_G_Ag', 'EWMA_Home_Pts',
    'EWMA_Away_Shots_For', 'EWMA_Away_ST_For', 'EWMA_Away_G_For', 'EWMA_Away_G_Ag', 'EWMA_Away_Pts',
    'Proj_xG_Home', 'Proj_xG_Away', 'Poisson_1', 'Poisson_X', 'Poisson_2', 'Poisson_O25'
]
df_final['Target'] = np.select([(df_final['FTHG'] > df_final['FTAG']), (df_final['FTHG'] == df_final['FTAG']), (df_final['FTHG'] < df_final['FTAG'])], [0, 1, 2])

#r probamos con el conjunto test, datos que no ha visto la red 
X_final = df_final[features]
y_final = df_final['Target']

X_train_super, y_train_super = X_final.iloc[:val_idx], y_final.iloc[:val_idx]
X_test_blind, y_test_blind = X_final.iloc[val_idx:], y_final.iloc[val_idx:]
df_test_blind = df_final.iloc[val_idx:].reset_index(drop=True)

scaler_final = StandardScaler()
X_train_super_sc = scaler_final.fit_transform(X_train_super)
X_test_blind_sc = scaler_final.transform(X_test_blind)
# configuramos la red con los mejores hiperparámetros según optuna 
xgb_final = XGBClassifier(n_estimators=best_params['xgb_n_est'], learning_rate=best_params['xgb_lr'], max_depth=3, random_state=42)
xgb_final.fit(X_train_super.values, y_train_super.values)
p_xgb_blind = xgb_final.predict_proba(X_test_blind.values)

nn_final = Sequential([
    Input(shape=(X_train_super.shape[1],)),
    Dense(32, activation='relu'), BatchNormalization(), Dropout(best_params['nn_dropout']),
    Dense(16, activation='relu'), BatchNormalization(), Dropout(best_params['nn_dropout']),
    Dense(3, activation='softmax')
])
nn_final.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=best_params['nn_lr']), loss='sparse_categorical_crossentropy')
es_final = EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True, verbose=0)
nn_final.fit(X_train_super_sc, y_train_super.values, validation_split=0.2, epochs=40, batch_size=32, callbacks=[es_final], verbose=0)

p_nn_blind = nn_final.predict(X_test_blind_sc, verbose=0)
p_ens_blind = (p_xgb_blind + p_nn_blind) / 2.0

# simulación financiera en el conunto de prueba
saldo_test = 0.0
apostado_test = 0
aciertos_test = 0

for i in range(len(df_test_blind)):
    row = df_test_blind.iloc[i]
    odds = [row['Odds_Home'], row['Odds_Draw'], row['Odds_Away']]
    probs = p_ens_blind[i]
    real_target = row['Target']
    
    odd_1X = (odds[0] * odds[1]) / (odds[0] + odds[1])
    odd_X2 = (odds[1] * odds[2]) / (odds[1] + odds[2])
    
    ev_1 = (probs[0] * odds[0]) - 1; ev_2 = (probs[2] * odds[2]) - 1
    ev_1X = ((probs[0] + probs[1]) * odd_1X) - 1; ev_X2 = ((probs[1] + probs[2]) * odd_X2) - 1
    
    bet = None

    # seleccionamos el umbral que recomienda optuna 
    if ev_1 > 0.08 and probs[0] > 0.45: bet = {'res': [0], 'odd': odds[0]}
    elif ev_1X > 0.08: bet = {'res': [0, 1], 'odd': odd_1X}
    elif ev_2 > 0.08 and probs[2] > 0.45: bet = {'res': [2], 'odd': odds[2]}
    elif ev_X2 > 0.08: bet = {'res': [1, 2], 'odd': odd_X2}
    
    if bet:
        apostado_test += 10
        if real_target in bet['res']: 
            saldo_test += (10 * bet['odd']) - 10
            aciertos_test += 1
        else: 
            saldo_test -= 10

roi_blind = (saldo_test / apostado_test) * 100 if apostado_test > 0 else 0
hit_rate_blind = (aciertos_test / (apostado_test/10)) * 100 if apostado_test > 0 else 0

print(f"\n=======================================")
print(f"📊 RESULTADOS ABSOLUTOS (TEST CIEGO 15%)")
print(f"📈 ROI FINAL REAL: {roi_blind:.2f}%")
print(f"🎯 Hit Rate:       {hit_rate_blind:.2f}%")
print(f"💶 Ganancia Neta:  {saldo_test:.2f}€")
print(f"📊 Total Apostado: {apostado_test:.2f}€")
print(f"=======================================")


📊 RESULTADOS ABSOLUTOS (TEST CIEGO 15%)
📈 ROI FINAL REAL: 16.85%
🎯 Hit Rate:       76.44%
💶 Ganancia Neta:  321.87€
📊 Total Apostado: 1910.00€


In [12]:
best_params

{'alpha': 0.31979862076204124,
 'xgb_n_est': 100,
 'xgb_lr': 0.0309862216249726,
 'nn_lr': 0.001683739014238699,
 'nn_dropout': 0.30918958604525215,
 'umbral_ev': 0.07320366290640293}

In [15]:
import joblib
# 1. Guardar el Escalador (Crucial para que los datos futuros se normalicen igual)
joblib.dump(scaler_final, 'EWMA_resultados_scaler.pkl')

# 2. Guardar el modelo XGBoost
xgb_final.save_model('EWMA_resultados_xgb.json')

# 3. Guardar la Red Neuronal
nn_final.save('EWMA_resultados_nn.keras')

Haré lo mismo para el mercado de goles.

In [16]:
import optuna
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler
from scipy.stats import poisson
import warnings
warnings.filterwarnings('ignore')

print("🏛️ INICIANDO LABORATORIO GOLES (O/U 2.5): 3-WAY SPLIT...")

df_base = df_crudo.copy()

# cuotas de goles 
if 'Avg>2.5' in df_base.columns:
    df_base['Odds_Over'] = df_base['Avg>2.5']
    df_base['Odds_Under'] = df_base['Avg<2.5']
elif 'B365>2.5' in df_base.columns:
    df_base['Odds_Over'] = df_base['B365>2.5']
    df_base['Odds_Under'] = df_base['B365<2.5']
else:
    print("⚠️ ADVERTENCIA: No se encontraron cuotas de goles (>2.5 / <2.5).")

# Limpiamos nulos en cuotas de goles
df_base = df_base.dropna(subset=['Odds_Over', 'Odds_Under']).reset_index(drop=True)

# orden cronológico
df_base['Date'] = pd.to_datetime(df_base['Date'])
df_base = df_base.sort_values('Date').reset_index(drop=True)

N = len(df_base)
train_idx = int(N * 0.70)
val_idx = int(N * 0.85)

def objetivo_goles_honesto(trial):
    alpha_opt = trial.suggest_float('alpha', 0.05, 0.40)
    xgb_n_est = trial.suggest_int('xgb_n_est', 100, 300, step=50)
    xgb_lr = trial.suggest_float('xgb_lr', 0.01, 0.05)
    nn_lr = trial.suggest_float('nn_lr', 0.001, 0.01, log=True)
    nn_dropout = trial.suggest_float('nn_dropout', 0.2, 0.5)
    umbral_ev = trial.suggest_float('umbral_ev', 0.04, 0.12) # Goles aceptan umbrales más altos
    
# EWMA
    df_temp = df_base.copy()
    
    home = df_temp[['Date', 'HomeTeam', 'HS', 'AS', 'HST', 'AST', 'FTHG', 'FTAG', 'Home_Pts', 'xG_Home', 'xG_Away']].rename(
        columns={'HomeTeam': 'Team', 'HS': 'Shots_For', 'AS': 'Shots_Ag', 'HST': 'ST_For', 'AST': 'ST_Ag', 
                 'FTHG': 'G_For', 'FTAG': 'G_Ag', 'Home_Pts': 'Pts', 'xG_Home': 'xG_For', 'xG_Away': 'xG_Ag'})
    away = df_temp[['Date', 'AwayTeam', 'AS', 'HS', 'AST', 'HST', 'FTAG', 'FTHG', 'Away_Pts', 'xG_Away', 'xG_Home']].rename(
        columns={'AwayTeam': 'Team', 'AS': 'Shots_For', 'HS': 'Shots_Ag', 'AST': 'ST_For', 'HST': 'ST_Ag', 
                 'FTAG': 'G_For', 'FTHG': 'G_Ag', 'Away_Pts': 'Pts', 'xG_Away': 'xG_For', 'xG_Home': 'xG_Ag'})
    
    combined = pd.concat([home, away]).sort_values(['Team', 'Date'])
    
    metricas = ['Shots_For', 'Shots_Ag', 'ST_For', 'ST_Ag', 'G_For', 'G_Ag', 'Pts', 'xG_For', 'xG_Ag']
    for m in metricas:
        combined[f'Form_{m}'] = combined.groupby('Team')[m].transform(lambda x: x.shift().ewm(alpha=alpha_opt, adjust=False).mean())
        
    cols_form = ['Date', 'Team'] + [f'Form_{m}' for m in metricas]
    df_temp = pd.merge(df_temp, combined[cols_form], left_on=['Date', 'HomeTeam'], right_on=['Date', 'Team'], how='left').rename(columns={f'Form_{m}': f'EWMA_Home_{m}' for m in metricas}).drop(columns=['Team'])
    df_temp = pd.merge(df_temp, combined[cols_form], left_on=['Date', 'AwayTeam'], right_on=['Date', 'Team'], how='left').rename(columns={f'Form_{m}': f'EWMA_Away_{m}' for m in metricas}).drop(columns=['Team'])
    
    df_temp.fillna(1.0, inplace=True)
    
    df_temp['Proj_xG_Home'] = (df_temp['EWMA_Home_xG_For'] + df_temp['EWMA_Away_xG_Ag']) / 2.0
    df_temp['Proj_xG_Away'] = (df_temp['EWMA_Away_xG_For'] + df_temp['EWMA_Home_xG_Ag']) / 2.0
    #matriz de poisson
    def calc_poisson(row):
        try:
            p_H = [poisson.pmf(k, max(0.1, row['Proj_xG_Home'])) for k in range(6)]
            p_A = [poisson.pmf(k, max(0.1, row['Proj_xG_Away'])) for k in range(6)]
            matriz = np.outer(p_H, p_A)
            return pd.Series([np.trace(matriz), 1 - (matriz[0:3, 0:3].sum())])
        except: return pd.Series([0.33, 0.5])

    df_temp[['Poisson_X', 'Poisson_O25']] = df_temp.apply(calc_poisson, axis=1)
    
    features = [
        'Elo_Diff', 'Elo_Home', 'Elo_Away', 
        'Odds_Home', 'Odds_Draw', 'Odds_Away', 
        'EWMA_Home_Shots_For', 'EWMA_Home_ST_For', 'EWMA_Home_G_For', 'EWMA_Home_G_Ag',
        'EWMA_Away_Shots_For', 'EWMA_Away_ST_For', 'EWMA_Away_G_For', 'EWMA_Away_G_Ag',
        'Proj_xG_Home', 'Proj_xG_Away', 'Poisson_X', 'Poisson_O25'
    ]
    
    # Target de goles 
    df_temp['Target_Goals'] = ((df_temp['FTHG'] + df_temp['FTAG']) > 2.5).astype(int)
    
    # Separación de datos 
    X = df_temp[features]
    y = df_temp['Target_Goals']
    
    X_train, y_train = X.iloc[:train_idx], y.iloc[:train_idx]
    X_val, y_val = X.iloc[train_idx:val_idx], y.iloc[train_idx:val_idx]
    df_val_real = df_temp.iloc[train_idx:val_idx].reset_index(drop=True) 
    
    scaler = StandardScaler()
    X_train_sc = scaler.fit_transform(X_train)
    X_val_sc = scaler.transform(X_val)
    
    # Entrenamiento con datos de train
    xgb_model = XGBClassifier(n_estimators=xgb_n_est, learning_rate=xgb_lr, max_depth=3, random_state=42, n_jobs=-1)
    xgb_model.fit(X_train.values, y_train.values)
    p_xgb = xgb_model.predict_proba(X_val.values)
    
    tf.get_logger().setLevel('ERROR')
    nn_model = Sequential([
        Input(shape=(X_train.shape[1],)),
        Dense(32, activation='relu'), BatchNormalization(), Dropout(nn_dropout),
        Dense(16, activation='relu'), BatchNormalization(), Dropout(nn_dropout),
        Dense(2, activation='softmax') # Solo 2 salidas para Goles
    ])
    nn_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=nn_lr), loss='sparse_categorical_crossentropy')
    es = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True, verbose=0)
    nn_model.fit(X_train_sc, y_train.values, validation_split=0.2, epochs=25, batch_size=32, callbacks=[es], verbose=0)
    
    p_nn = nn_model.predict(X_val_sc, verbose=0)
    p_ens = (p_xgb + p_nn) / 2.0
    
    # Evaluación en validación
    saldo = 0.0
    apostado = 0
    
    for i in range(len(df_val_real)):
        row = df_val_real.iloc[i]
        odd_Ov = row['Odds_Over']
        odd_Un = row['Odds_Under']
        
        # p_ens devuelve [Prob_Under, Prob_Over]
        p_Un, p_Ov = p_ens[i]
        real_target = row['Target_Goals']
        
        ev_Ov = (p_Ov * odd_Ov) - 1
        ev_Un = (p_Un * odd_Un) - 1
        
        bet = None
        if ev_Ov > umbral_ev: bet = {'res': 1, 'odd': odd_Ov}
        elif ev_Un > umbral_ev: bet = {'res': 0, 'odd': odd_Un}
        
        if bet:
            apostado += 10
            if real_target == bet['res']: saldo += (10 * bet['odd']) - 10
            else: saldo -= 10
            
    if apostado < 100: return -100.0 
    return (saldo / apostado) * 100

optuna.logging.set_verbosity(optuna.logging.WARNING)
study_goles = optuna.create_study(direction='maximize')
study_goles.optimize(objetivo_goles_honesto, n_trials=15)

print("\n🏆 ¡OPTUNA GOLES TERMINÓ LA BÚSQUEDA!")
best_params = study_goles.best_params
print(f"💰 MEJOR ROI en Validación: {study_goles.best_value:.2f}%")
print(f"👉 Alfa EWMA Elegido para Goles: {best_params['alpha']:.4f}")

🏛️ INICIANDO LABORATORIO GOLES (O/U 2.5): 3-WAY SPLIT...

🏆 ¡OPTUNA GOLES TERMINÓ LA BÚSQUEDA!
💰 MEJOR ROI en Validación: 7.14%
👉 Alfa EWMA Elegido para Goles: 0.2108


In [18]:
best_params

{'alpha': 0.210768349941788,
 'xgb_n_est': 200,
 'xgb_lr': 0.020373514666431686,
 'nn_lr': 0.001168533165400629,
 'nn_dropout': 0.27313312627997616,
 'umbral_ev': 0.08198767966916207}

In [17]:
df_final = df_base.copy()
home = df_final[['Date', 'HomeTeam', 'HS', 'AS', 'HST', 'AST', 'FTHG', 'FTAG', 'Home_Pts', 'xG_Home', 'xG_Away']].rename(columns={'HomeTeam': 'Team', 'HS': 'Shots_For', 'AS': 'Shots_Ag', 'HST': 'ST_For', 'AST': 'ST_Ag', 'FTHG': 'G_For', 'FTAG': 'G_Ag', 'Home_Pts': 'Pts', 'xG_Home': 'xG_For', 'xG_Away': 'xG_Ag'})
away = df_final[['Date', 'AwayTeam', 'AS', 'HS', 'AST', 'HST', 'FTAG', 'FTHG', 'Away_Pts', 'xG_Away', 'xG_Home']].rename(columns={'AwayTeam': 'Team', 'AS': 'Shots_For', 'HS': 'Shots_Ag', 'AST': 'ST_For', 'HST': 'ST_Ag', 'FTAG': 'G_For', 'FTHG': 'G_Ag', 'Away_Pts': 'Pts', 'xG_Away': 'xG_For', 'xG_Home': 'xG_Ag'})
combined = pd.concat([home, away]).sort_values(['Team', 'Date'])

metricas = ['Shots_For', 'Shots_Ag', 'ST_For', 'ST_Ag', 'G_For', 'G_Ag', 'Pts', 'xG_For', 'xG_Ag']
#usamos el alpha recomendado por optuna
for m in metricas: combined[f'Form_{m}'] = combined.groupby('Team')[m].transform(lambda x: x.shift().ewm(alpha=best_params['alpha'], adjust=False).mean())
cols_form = ['Date', 'Team'] + [f'Form_{m}' for m in metricas]

df_final = pd.merge(df_final, combined[cols_form], left_on=['Date', 'HomeTeam'], right_on=['Date', 'Team'], how='left').rename(columns={f'Form_{m}': f'EWMA_Home_{m}' for m in metricas}).drop(columns=['Team'])
df_final = pd.merge(df_final, combined[cols_form], left_on=['Date', 'AwayTeam'], right_on=['Date', 'Team'], how='left').rename(columns={f'Form_{m}': f'EWMA_Away_{m}' for m in metricas}).drop(columns=['Team'])
df_final.fillna(1.0, inplace=True)

df_final['Proj_xG_Home'] = (df_final['EWMA_Home_xG_For'] + df_final['EWMA_Away_xG_Ag']) / 2.0
df_final['Proj_xG_Away'] = (df_final['EWMA_Away_xG_For'] + df_final['EWMA_Home_xG_Ag']) / 2.0

def calc_poisson_final(row):
    try:
        p_H = [poisson.pmf(k, max(0.1, row['Proj_xG_Home'])) for k in range(6)]
        p_A = [poisson.pmf(k, max(0.1, row['Proj_xG_Away'])) for k in range(6)]
        matriz = np.outer(p_H, p_A)
        return pd.Series([np.trace(matriz), 1 - (matriz[0:3, 0:3].sum())])
    except: return pd.Series([0.33, 0.5])

df_final[['Poisson_X', 'Poisson_O25']] = df_final.apply(calc_poisson_final, axis=1)

features = [
    'Elo_Diff', 'Elo_Home', 'Elo_Away', 'Odds_Home', 'Odds_Draw', 'Odds_Away',
    'EWMA_Home_Shots_For', 'EWMA_Home_ST_For', 'EWMA_Home_G_For', 'EWMA_Home_G_Ag',
    'EWMA_Away_Shots_For', 'EWMA_Away_ST_For', 'EWMA_Away_G_For', 'EWMA_Away_G_Ag',
    'Proj_xG_Home', 'Proj_xG_Away', 'Poisson_X', 'Poisson_O25'
]
df_final['Target_Goals'] = ((df_final['FTHG'] + df_final['FTAG']) > 2.5).astype(int)

X_final = df_final[features]
y_final = df_final['Target_Goals']

X_train_super, y_train_super = X_final.iloc[:val_idx], y_final.iloc[:val_idx]
X_test_blind, y_test_blind = X_final.iloc[val_idx:], y_final.iloc[val_idx:]
df_test_blind = df_final.iloc[val_idx:].reset_index(drop=True)

scaler_final = StandardScaler()
X_train_super_sc = scaler_final.fit_transform(X_train_super)
X_test_blind_sc = scaler_final.transform(X_test_blind)

#configuramos la red y xgboost con los mejores hiperparámteros  
xgb_final = XGBClassifier(n_estimators=best_params['xgb_n_est'], learning_rate=best_params['xgb_lr'], max_depth=3, random_state=42)
xgb_final.fit(X_train_super.values, y_train_super.values)
p_xgb_blind = xgb_final.predict_proba(X_test_blind.values)

nn_final = Sequential([
    Input(shape=(X_train_super.shape[1],)),
    Dense(32, activation='relu'), BatchNormalization(), Dropout(best_params['nn_dropout']),
    Dense(16, activation='relu'), BatchNormalization(), Dropout(best_params['nn_dropout']),
    Dense(2, activation='softmax')
])
nn_final.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=best_params['nn_lr']), loss='sparse_categorical_crossentropy')
es_final = EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True, verbose=0)
nn_final.fit(X_train_super_sc, y_train_super.values, validation_split=0.2, epochs=40, batch_size=32, callbacks=[es_final], verbose=0)

p_nn_blind = nn_final.predict(X_test_blind_sc, verbose=0)
p_ens_blind = (p_xgb_blind + p_nn_blind) / 2.0

# Simulación financiera en el conjunto de prueba 
saldo_test = 0.0
apostado_test = 0
aciertos_test = 0

for i in range(len(df_test_blind)):
    row = df_test_blind.iloc[i]
    odd_Ov = row['Odds_Over']
    odd_Un = row['Odds_Under']
    
    p_Un, p_Ov = p_ens_blind[i]
    real_target = row['Target_Goals']
    
    ev_Ov = (p_Ov * odd_Ov) - 1
    ev_Un = (p_Un * odd_Un) - 1
    
    bet = None
    if ev_Ov > best_params['umbral_ev']: bet = {'res': 1, 'odd': odd_Ov}
    elif ev_Un > best_params['umbral_ev']: bet = {'res': 0, 'odd': odd_Un}
    
    if bet:
        apostado_test += 10
        if real_target == bet['res']: 
            saldo_test += (10 * bet['odd']) - 10
            aciertos_test += 1
        else: 
            saldo_test -= 10

roi_blind = (saldo_test / apostado_test) * 100 if apostado_test > 0 else 0
hit_rate_blind = (aciertos_test / (apostado_test/10)) * 100 if apostado_test > 0 else 0

print(f"\n=======================================")
print(f"📊 RESULTADOS ABSOLUTOS (TEST CIEGO GOLES)")
print(f"📈 ROI FINAL REAL: {roi_blind:.2f}%")
print(f"🎯 Hit Rate:       {hit_rate_blind:.2f}%")
print(f"💶 Ganancia Neta:  {saldo_test:.2f}€")
print(f"📊 Total Apostado: {apostado_test:.2f}€")
print(f"=======================================")


📊 RESULTADOS ABSOLUTOS (TEST CIEGO GOLES)
📈 ROI FINAL REAL: -4.99%
🎯 Hit Rate:       42.07%
💶 Ganancia Neta:  -72.40€
📊 Total Apostado: 1450.00€


El resultado no es el mejor en el modelo de goles, el siguiente paso es hacer un código que genere parlays únicamente de resultados .

In [22]:
import pandas as pd
import numpy as np
import joblib
from tensorflow.keras.models import load_model
import xgboost as xgb
from scipy.stats import poisson
import itertools
import os
import warnings
warnings.filterwarnings('ignore')

print("==========================================================")
print("🏦 ORÁCULO DE PRODUCCIÓN: PARLAYS DE RESULTADOS (EWMA) 🏦")
print("==========================================================")

# ==========================================
# 1. CONFIGURACIÓN
# ==========================================
MI_BANKROLL_TOTAL = 1000.0  
KELLY_FRACCIONAL = 0.25     
UMBRAL_PARLAY = 0.05        # Buscamos al menos 5% de EV en combinadas
MAX_SELECCIONES = 3         # Dobles o Triples (no más para evitar loterías)
OPT_ALPHA = 0.1025          # Nuestro Alfa de Oro descubierto por Optuna

# ==========================================
# 2. CARGAR EL MODELO BLINDADO
# ==========================================
print("🧠 Cargando Inteligencia Artificial EWMA...")
try:
    os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3' 
    scaler = joblib.load('EWMA_resultados_scaler.pkl')
    modelo_xgb = xgb.XGBClassifier()
    modelo_xgb.load_model('EWMA_resultados_xgb.json')
    modelo_nn = load_model('EWMA_resultados_nn.keras')
    print("   ✅ Cerebro de Resultados cargado y listo para operar.")
except Exception as e:
    print(f"   ❌ ERROR FATAL AL CARGAR MODELOS: {e}")

# ==========================================
# 3. FUNCIONES DE EXTRACCIÓN DE DATOS
# ==========================================
metricas_base = ['Shots_For', 'Shots_Ag', 'ST_For', 'ST_Ag', 'G_For', 'G_Ag', 'Pts', 'xG_For', 'xG_Ag']

def obtener_elo_actual(equipo, df_hist):
    df_h = df_hist[df_hist['HomeTeam'] == equipo][['Date', 'Elo_Home']].rename(columns={'Elo_Home': 'Elo'})
    df_a = df_hist[df_hist['AwayTeam'] == equipo][['Date', 'Elo_Away']].rename(columns={'Elo_Away': 'Elo'})
    df_equipo = pd.concat([df_h, df_a]).sort_values('Date')
    return df_equipo['Elo'].iloc[-1] if not df_equipo.empty else 1500

def obtener_forma_ewma(equipo, df_hist, alpha):
    df_h = df_hist[df_hist['HomeTeam'] == equipo][['Date', 'HS', 'AS', 'HST', 'AST', 'FTHG', 'FTAG', 'Home_Pts', 'xG_Home', 'xG_Away']].rename(
        columns={'HS': 'Shots_For', 'AS': 'Shots_Ag', 'HST': 'ST_For', 'AST': 'ST_Ag', 'FTHG': 'G_For', 'FTAG': 'G_Ag', 'Home_Pts': 'Pts', 'xG_Home': 'xG_For', 'xG_Away': 'xG_Ag'})
    df_a = df_hist[df_hist['AwayTeam'] == equipo][['Date', 'AS', 'HS', 'AST', 'HST', 'FTAG', 'FTHG', 'Away_Pts', 'xG_Away', 'xG_Home']].rename(
        columns={'AS': 'Shots_For', 'HS': 'Shots_Ag', 'AST': 'ST_For', 'HST': 'ST_Ag', 'FTAG': 'G_For', 'FTHG': 'G_Ag', 'Away_Pts': 'Pts', 'xG_Away': 'xG_For', 'xG_Home': 'xG_Ag'})
    
    df_equipo = pd.concat([df_h, df_a]).sort_values('Date')
    if df_equipo.empty: return None
    
    # Calculamos EWMA hasta el último partido jugado (sin shift)
    forma = {}
    for m in metricas_base:
        serie_ewma = df_equipo[m].ewm(alpha=alpha, adjust=False).mean()
        forma[m] = serie_ewma.iloc[-1]
    return forma

# ==========================================
# 4. PARTIDOS A APOSTAR HOY (Tus cuotas)
# ==========================================
partidos_jornada = [
    {'Local': 'Celta', 'Visita': 'Alaves', 'Odd_1': 1.93, 'Odd_X': 3.27, 'Odd_2': 4.23},
    {'Local': 'Ath Bilbao', 'Visita': "Betis", 'Odd_1': 2.09, 'Odd_X': 3.43, 'Odd_2': 3.44},
    {'Local': 'Real Madrid', 'Visita': 'Ath Madrid', 'Odd_1': 1.82, 'Odd_X': 4.00, 'Odd_2': 3.86},
    {'Local': 'Roma', 'Visita': 'Lecce', 'Odd_1':1.46 , 'Odd_X': 4.31, 'Odd_2': 6.93},
    {'Local': 'Fiorentina', 'Visita': 'Inter', 'Odd_1': 5.05, 'Odd_X': 4.12, 'Odd_2': 1.62},
    {'Local': 'Mainz', 'Visita': 'Ein Frankfurt', 'Odd_1': 2.15, 'Odd_X': 3.52, 'Odd_2': 3.22},
    {'Local': 'St Pauli', 'Visita': 'Freiburg', 'Odd_1': 2.61, 'Odd_X': 3.04, 'Odd_2': 2.87},
    {'Local': 'Augsburg', 'Visita': 'Stuttgart', 'Odd_1': 3.28, 'Odd_X': 3.77, 'Odd_2': 2.04},
    {'Local': 'Marseille', 'Visita': 'Lille', 'Odd_1': 1.83, 'Odd_X': 3.72, 'Odd_2': 4.11},
    {'Local': 'Paris FC', 'Visita': 'Le Havre', 'Odd_1': 1.98, 'Odd_X': 3.27, 'Odd_2': 4.01},
    {'Local': 'Rennes', 'Visita': 'Metz', 'Odd_1': 1.30, 'Odd_X': 5.68, 'Odd_2': 8.77},
    {'Local': 'Nantes', 'Visita': 'Strasbourg', 'Odd_1': 3.65, 'Odd_X': 3.57, 'Odd_2': 1.98},
]

# ==========================================
# 5. EL ESCÁNER DE PARLAYS
# ==========================================
print("\n🔍 Analizando jornada y buscando combinaciones óptimas...")
candidatos_parlay = []

for p in partidos_jornada:
    local, visita = p['Local'], p['Visita']
    odd_L, odd_E, odd_V = p['Odd_1'], p['Odd_X'], p['Odd_2']
    
    try:
        elo_H = obtener_elo_actual(local, df_crudo)
        elo_A = obtener_elo_actual(visita, df_crudo)
        elo_diff = (elo_H + 50) - elo_A
        
        forma_H = obtener_forma_ewma(local, df_crudo, OPT_ALPHA)
        forma_A = obtener_forma_ewma(visita, df_crudo, OPT_ALPHA)
        
        if forma_H is None or forma_A is None: continue
            
        proj_xg_home = (forma_H['xG_For'] + forma_A['xG_Ag']) / 2.0
        proj_xg_away = (forma_A['xG_For'] + forma_H['xG_Ag']) / 2.0
        
        p_H = [poisson.pmf(k, max(0.1, proj_xg_home)) for k in range(6)]
        p_A = [poisson.pmf(k, max(0.1, proj_xg_away)) for k in range(6)]
        matriz = np.outer(p_H, p_A)
        
        input_dict = {
            'Elo_Diff': elo_diff, 'Elo_Home': elo_H, 'Elo_Away': elo_A,
            'Odds_Home': odd_L, 'Odds_Draw': odd_E, 'Odds_Away': odd_V,
            'EWMA_Home_Shots_For': forma_H['Shots_For'], 'EWMA_Home_ST_For': forma_H['ST_For'], 
            'EWMA_Home_G_For': forma_H['G_For'], 'EWMA_Home_G_Ag': forma_H['G_Ag'], 'EWMA_Home_Pts': forma_H['Pts'],
            'EWMA_Away_Shots_For': forma_A['Shots_For'], 'EWMA_Away_ST_For': forma_A['ST_For'], 
            'EWMA_Away_G_For': forma_A['G_For'], 'EWMA_Away_G_Ag': forma_A['G_Ag'], 'EWMA_Away_Pts': forma_A['Pts'],
            'Proj_xG_Home': proj_xg_home, 'Proj_xG_Away': proj_xg_away,
            'Poisson_1': np.tril(matriz, -1).sum(), 'Poisson_X': np.trace(matriz), 
            'Poisson_2': np.triu(matriz, 1).sum(), 'Poisson_O25': 1 - (matriz[0:3, 0:3].sum())
        }
        
        # El orden estricto de las 22 variables
        features_order = [
            'Elo_Diff', 'Elo_Home', 'Elo_Away', 'Odds_Home', 'Odds_Draw', 'Odds_Away',
            'EWMA_Home_Shots_For', 'EWMA_Home_ST_For', 'EWMA_Home_G_For', 'EWMA_Home_G_Ag', 'EWMA_Home_Pts',
            'EWMA_Away_Shots_For', 'EWMA_Away_ST_For', 'EWMA_Away_G_For', 'EWMA_Away_G_Ag', 'EWMA_Away_Pts',
            'Proj_xG_Home', 'Proj_xG_Away', 'Poisson_1', 'Poisson_X', 'Poisson_2', 'Poisson_O25'
        ]
        
        X_input = pd.DataFrame([input_dict])[features_order]
        
        # Fusión Híbrida
        prob_xgb = modelo_xgb.predict_proba(X_input.values)[0]
        prob_nn = modelo_nn.predict(scaler.transform(X_input), verbose=0)[0]
        p_L, p_E, p_V = (prob_xgb + prob_nn) / 2.0
        
        odd_1X = (odd_L * odd_E) / (odd_L + odd_E)
        odd_X2 = (odd_E * odd_V) / (odd_E + odd_V)
        
        # Recolectar "Ladrillos" para Parlays (Probabilidad > 60%)
        if p_L + p_E > 0.60: candidatos_parlay.append({'partido': f"{local} vs {visita}", 'mercado': f"{local} (1X)", 'prob': p_L + p_E, 'odd': odd_1X})
        if p_E + p_V > 0.60: candidatos_parlay.append({'partido': f"{local} vs {visita}", 'mercado': f"{visita} (X2)", 'prob': p_E + p_V, 'odd': odd_X2})
        if p_L > 0.55: candidatos_parlay.append({'partido': f"{local} vs {visita}", 'mercado': f"Gana {local}", 'prob': p_L, 'odd': odd_L})
        if p_V > 0.55: candidatos_parlay.append({'partido': f"{local} vs {visita}", 'mercado': f"Gana {visita}", 'prob': p_V, 'odd': odd_V})

    except Exception as e:
        print(f"   ⚠️ No se pudo analizar {local} vs {visita}: {e}")
        continue

# ==========================================
# 6. CONSTRUCCIÓN COMBINATORIA
# ==========================================
parlays_encontrados = []
inversion_total = 0.0

for k in range(2, MAX_SELECCIONES + 1):
    if len(candidatos_parlay) < k: break
        
    for combo in itertools.combinations(candidatos_parlay, k):
        partidos_en_ticket = [apuesta['partido'] for apuesta in combo]
        if len(set(partidos_en_ticket)) < len(partidos_en_ticket): continue # Evita el mismo partido 2 veces
            
        prob_comb = np.prod([apuesta['prob'] for apuesta in combo])
        odd_comb = np.prod([apuesta['odd'] for apuesta in combo])
        ev_comb = (prob_comb * odd_comb) - 1
        
        if ev_comb > UMBRAL_PARLAY:
            parlays_encontrados.append({'legs': k, 'combo': combo, 'prob': prob_comb, 'odd': odd_comb, 'ev': ev_comb})

if parlays_encontrados:
    print("\n" + "="*56)
    print(f"🔥 MEJORES PARLAYS DE RESULTADOS (Top 4 por Valor)")
    print("="*56)
    
    parlays_encontrados = sorted(parlays_encontrados, key=lambda x: x['ev'], reverse=True)
    
    for i, parlay in enumerate(parlays_encontrados[:4]):
        max_stake_pct = 0.04 if parlay['legs'] == 2 else 0.015 
        stake = min(max(0, (parlay['ev'] / (parlay['odd'] - 1)) * KELLY_FRACCIONAL), max_stake_pct)
        monto = MI_BANKROLL_TOTAL * stake
        
        if monto > 0:
            inversion_total += monto
            print(f"\n🎟️ TICKET #{i+1} ({parlay['legs']} Selecciones) | Cuota: {parlay['odd']:.2f}")
            for j, leg in enumerate(parlay['combo']):
                print(f"   [{j+1}] {leg['partido']} -> {leg['mercado']} (@{leg['odd']:.2f})")
            print(f"   📈 EV: +{parlay['ev']*100:.1f}% | 💰 INVERTIR: €{monto:.2f} ({stake*100:.2f}% Bankroll)")

print("\n" + "="*56)
print(f"💵 INVERSIÓN TOTAL RECOMENDADA: €{inversion_total:.2f}")
print("="*56)

🏦 ORÁCULO DE PRODUCCIÓN: PARLAYS DE RESULTADOS (EWMA) 🏦
🧠 Cargando Inteligencia Artificial EWMA...
   ✅ Cerebro de Resultados cargado y listo para operar.

🔍 Analizando jornada y buscando combinaciones óptimas...

🔥 MEJORES PARLAYS DE RESULTADOS (Top 4 por Valor)

🎟️ TICKET #1 (3 Selecciones) | Cuota: 4.86
   [1] Real Madrid vs Ath Madrid -> Gana Real Madrid (@1.82)
   [2] Roma vs Lecce -> Gana Roma (@1.46)
   [3] Marseille vs Lille -> Gana Marseille (@1.83)
   📈 EV: +19.5% | 💰 INVERTIR: €12.63 (1.26% Bankroll)

🎟️ TICKET #2 (3 Selecciones) | Cuota: 5.40
   [1] Real Madrid vs Ath Madrid -> Gana Real Madrid (@1.82)
   [2] Fiorentina vs Inter -> Gana Inter (@1.62)
   [3] Marseille vs Lille -> Gana Marseille (@1.83)
   📈 EV: +17.5% | 💰 INVERTIR: €9.94 (0.99% Bankroll)

🎟️ TICKET #3 (3 Selecciones) | Cuota: 4.30
   [1] Real Madrid vs Ath Madrid -> Gana Real Madrid (@1.82)
   [2] Roma vs Lecce -> Gana Roma (@1.46)
   [3] Fiorentina vs Inter -> Gana Inter (@1.62)
   📈 EV: +17.1% | 💰 INVERTIR

### Arquitectura LSTM

In [5]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, LSTM, Concatenate, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import StandardScaler

print("🧠 INICIANDO ARQUITECTURA LSTM DE DOBLE RAMA...")

# =======================================================
# 1. PARÁMETROS DEL TENSOR
# =======================================================
TIMESTEPS = 5  # Memoria de los últimos 5 partidos
df_base = df_crudo.copy()
df_base['Date'] = pd.to_datetime(df_base['Date'])
df_base = df_base.sort_values('Date').reset_index(drop=True)

# =======================================================
# 2. SEPARAR EL HISTORIAL DE CADA EQUIPO
# =======================================================
print("   🔄 Extrayendo secuencias temporales por equipo...")
# Homogeneizamos los datos para tener el "Punto de vista" de cada equipo
home = df_base[['Date', 'HomeTeam', 'FTHG', 'FTAG', 'HS', 'AS', 'HST', 'AST', 'xG_Home', 'xG_Away']].rename(
    columns={'HomeTeam': 'Team', 'FTHG': 'GF', 'FTAG': 'GC', 'HS': 'SF', 'AS': 'SC', 'HST': 'STF', 'AST': 'STC', 'xG_Home': 'xGF', 'xG_Away': 'xGC'})
away = df_base[['Date', 'AwayTeam', 'FTAG', 'FTHG', 'AS', 'HS', 'AST', 'HST', 'xG_Away', 'xG_Home']].rename(
    columns={'AwayTeam': 'Team', 'FTAG': 'GF', 'FTHG': 'GC', 'AS': 'SF', 'HS': 'SC', 'AST': 'STF', 'HST': 'STC', 'xG_Away': 'xGF', 'xG_Home': 'xGC'})

historial_equipos = pd.concat([home, away]).sort_values(['Team', 'Date']).reset_index(drop=True)

# Variables secuenciales que la LSTM va a procesar
seq_features = ['GF', 'GC', 'SF', 'SC', 'STF', 'STC', 'xGF', 'xGC']
num_seq_features = len(seq_features)

# =======================================================
# 3. GENERADOR DE TENSORES 3D
# =======================================================
print("   🧊 Construyendo matrices 3D...")
X_seq_home = []
X_seq_away = []
X_static = []
y_target = []
indices_validos = []

# Agrupamos por equipo para búsquedas rápidas
dict_historial = {equipo: datos for equipo, datos in historial_equipos.groupby('Team')}

for i in range(len(df_base)):
    row = df_base.iloc[i]
    fecha = row['Date']
    local = row['HomeTeam']
    visita = row['AwayTeam']
    
    # Extraer historial estricto ANTERIOR a la fecha del partido
    hist_L = dict_historial[local]
    hist_L = hist_L[hist_L['Date'] < fecha].tail(TIMESTEPS)
    
    hist_V = dict_historial[visita]
    hist_V = hist_V[hist_V['Date'] < fecha].tail(TIMESTEPS)
    
    # Solo procesamos si ambos equipos tienen al menos 'TIMESTEPS' partidos previos
    if len(hist_L) == TIMESTEPS and len(hist_V) == TIMESTEPS:
        X_seq_home.append(hist_L[seq_features].values)
        X_seq_away.append(hist_V[seq_features].values)
        
        # Datos estáticos del partido actual
        elo_diff = (row['Elo_Home'] + 50) - row['Elo_Away']
        X_static.append([elo_diff, row['Odds_Home'], row['Odds_Draw'], row['Odds_Away']])
        
        # Target 1X2
        if row['FTHG'] > row['FTAG']: target = 0
        elif row['FTHG'] == row['FTAG']: target = 1
        else: target = 2
        y_target.append(target)
        
        indices_validos.append(i)

# Convertir a arrays de NumPy
X_seq_home = np.array(X_seq_home)    # Forma: (N, 5, 8)
X_seq_away = np.array(X_seq_away)    # Forma: (N, 5, 8)
X_static = np.array(X_static)        # Forma: (N, 4)
y_target = np.array(y_target)        # Forma: (N,)
df_lstm_validos = df_base.iloc[indices_validos].reset_index(drop=True)

print(f"   ✅ Tensores listos. Partidos procesables: {len(X_seq_home)}")

# =======================================================
# 4. SPLIT (TRAIN / TEST) CRONOLÓGICO
# =======================================================
cutoff = int(len(X_seq_home) * 0.85) # 85% Train, 15% Test Ciego

X_H_train, X_H_test = X_seq_home[:cutoff], X_seq_home[cutoff:]
X_A_train, X_A_test = X_seq_away[:cutoff], X_seq_away[cutoff:]
X_S_train, X_S_test = X_static[:cutoff], X_static[cutoff:]
y_train, y_test = y_target[:cutoff], y_target[cutoff:]
df_test = df_lstm_validos.iloc[cutoff:].reset_index(drop=True)

# Escalar datos estáticos (El tensor secuencial suele no escalarse o escalarse por timestep)
scaler_static = StandardScaler()
X_S_train_sc = scaler_static.fit_transform(X_S_train)
X_S_test_sc = scaler_static.transform(X_S_test)

# =======================================================
# 5. ARQUITECTURA DE LA RED SIAMESA (KERAS FUNCTIONAL API)
# =======================================================
print("   ⚙️ Ensamblando el cerebro LSTM...")
tf.get_logger().setLevel('ERROR')

# Rama 1: Historial del Local
input_H = Input(shape=(TIMESTEPS, num_seq_features), name="Historial_Local")
lstm_H = LSTM(32, activation='tanh', return_sequences=False)(input_H)
lstm_H = BatchNormalization()(lstm_H)
lstm_H = Dropout(0.3)(lstm_H)

# Rama 2: Historial de la Visita
input_A = Input(shape=(TIMESTEPS, num_seq_features), name="Historial_Visita")
lstm_A = LSTM(32, activation='tanh', return_sequences=False)(input_A)
lstm_A = BatchNormalization()(lstm_A)
lstm_A = Dropout(0.3)(lstm_A)

# Rama 3: Datos Estáticos (Cuotas, ELO)
input_S = Input(shape=(X_static.shape[1],), name="Datos_Estaticos")
dense_S = Dense(16, activation='relu')(input_S)

# Fusión
concat = Concatenate()([lstm_H, lstm_A, dense_S])
x = Dense(32, activation='relu')(concat)
x = BatchNormalization()(x)
x = Dropout(0.3)(x)
output = Dense(3, activation='softmax', name="Prediccion_1X2")(x)

modelo_lstm = Model(inputs=[input_H, input_A, input_S], outputs=output)
modelo_lstm.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.002), 
                    loss='sparse_categorical_crossentropy', 
                    metrics=['accuracy'])

# =======================================================
# 6. ENTRENAMIENTO
# =======================================================
print("   🚀 Entrenando modelo (Esto puede tomar un minuto)...")
es = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=0)

modelo_lstm.fit(
    [X_H_train, X_A_train, X_S_train_sc], y_train,
    validation_split=0.15,
    epochs=50,
    batch_size=32,
    callbacks=[es],
    verbose=1 # Lo dejamos en 1 para que veas el progreso de la LSTM
)

print("\n✅ Entrenamiento Finalizado. Red Siamesa LSTM lista.")

🧠 INICIANDO ARQUITECTURA LSTM DE DOBLE RAMA...
   🔄 Extrayendo secuencias temporales por equipo...
   🧊 Construyendo matrices 3D...
   ✅ Tensores listos. Partidos procesables: 8119
   ⚙️ Ensamblando el cerebro LSTM...
   🚀 Entrenando modelo (Esto puede tomar un minuto)...
Epoch 1/50
184/184 ━━━━━━━━━━━━━━━━━━━━ 12s 18ms/step - accuracy: 0.4443 - loss: 1.2008 - val_accuracy: 0.5415 - val_loss: 0.9719
Epoch 2/50
184/184 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - accuracy: 0.4910 - loss: 1.0444 - val_accuracy: 0.5434 - val_loss: 0.9709
Epoch 3/50
184/184 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.5081 - loss: 1.0058 - val_accuracy: 0.5357 - val_loss: 0.9693
Epoch 4/50
184/184 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.5246 - loss: 0.9911 - val_accuracy: 0.5386 - val_loss: 0.9681
Epoch 5/50
184/184 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - accuracy: 0.5253 - loss: 0.9864 - val_accuracy: 0.5434 - val_loss: 0.9686
Epoch 6/50
184/184 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.5304 - loss: 0

In [7]:
import optuna
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, LSTM, Concatenate, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# =======================================================
# 1. PREPARACIÓN BASE
# =======================================================
df_base = df_crudo.copy()
df_base['Date'] = pd.to_datetime(df_base['Date'])
df_base = df_base.sort_values('Date').reset_index(drop=True)

home = df_base[['Date', 'HomeTeam', 'FTHG', 'FTAG', 'HS', 'AS', 'HST', 'AST', 'xG_Home', 'xG_Away']].rename(
    columns={'HomeTeam': 'Team', 'FTHG': 'GF', 'FTAG': 'GC', 'HS': 'SF', 'AS': 'SC', 'HST': 'STF', 'AST': 'STC', 'xG_Home': 'xGF', 'xG_Away': 'xGC'})
away = df_base[['Date', 'AwayTeam', 'FTAG', 'FTHG', 'AS', 'HS', 'AST', 'HST', 'xG_Away', 'xG_Home']].rename(
    columns={'AwayTeam': 'Team', 'FTAG': 'GF', 'FTHG': 'GC', 'AS': 'SF', 'HS': 'SC', 'AST': 'STF', 'HST': 'STC', 'xG_Away': 'xGF', 'xG_Home': 'xGC'})

historial_equipos = pd.concat([home, away]).sort_values(['Team', 'Date']).reset_index(drop=True)
seq_features = ['GF', 'GC', 'SF', 'SC', 'STF', 'STC', 'xGF', 'xGC']
dict_historial = {equipo: datos for equipo, datos in historial_equipos.groupby('Team')}

def objetivo_lstm_honesto(trial):
    # --- 1. HIPERPARÁMETROS ---
    opt_timesteps = trial.suggest_int('timesteps', 3, 8) 
    opt_lstm_units = trial.suggest_categorical('lstm_units', [16, 32])
    opt_dense_units = trial.suggest_categorical('dense_units', [16, 32])
    opt_lr = trial.suggest_float('lr', 1e-4, 5e-3, log=True)
    opt_dropout = trial.suggest_float('dropout', 0.2, 0.5)
    opt_umbral_ev = trial.suggest_float('umbral_ev', 0.05, 0.12) 
    
    # --- 2. CONSTRUCCIÓN DE TENSORES 3D ---
    X_seq_home, X_seq_away, X_static, y_target, indices_validos = [], [], [], [], []
    
    for i in range(len(df_base)):
        row = df_base.iloc[i]
        fecha, local, visita = row['Date'], row['HomeTeam'], row['AwayTeam']
        
        hist_L = dict_historial[local]
        hist_L = hist_L[hist_L['Date'] < fecha].tail(opt_timesteps)
        
        hist_V = dict_historial[visita]
        hist_V = hist_V[hist_V['Date'] < fecha].tail(opt_timesteps)
        
        if len(hist_L) == opt_timesteps and len(hist_V) == opt_timesteps:
            X_seq_home.append(hist_L[seq_features].values)
            X_seq_away.append(hist_V[seq_features].values)
            
            elo_diff = (row['Elo_Home'] + 50) - row['Elo_Away']
            X_static.append([elo_diff, row['Odds_Home'], row['Odds_Draw'], row['Odds_Away']])
            
            if row['FTHG'] > row['FTAG']: target = 0
            elif row['FTHG'] == row['FTAG']: target = 1
            else: target = 2
            
            y_target.append(target)
            indices_validos.append(i)
            
    X_seq_home = np.array(X_seq_home)
    X_seq_away = np.array(X_seq_away)
    X_static = np.array(X_static)
    y_target = np.array(y_target)
    df_validos = df_base.iloc[indices_validos].reset_index(drop=True)
    
    # --- 3. SPLIT DE 3 VÍAS (Cortes Dinámicos) ---
    N_tensors = len(X_seq_home)
    train_cutoff = int(N_tensors * 0.70)
    val_cutoff = int(N_tensors * 0.85)
    
    X_H_train, X_H_val = X_seq_home[:train_cutoff], X_seq_home[train_cutoff:val_cutoff]
    X_A_train, X_A_val = X_seq_away[:train_cutoff], X_seq_away[train_cutoff:val_cutoff]
    X_S_train, X_S_val = X_static[:train_cutoff], X_static[train_cutoff:val_cutoff]
    y_train, y_val = y_target[:train_cutoff], y_target[train_cutoff:val_cutoff]
    df_val_real = df_validos.iloc[train_cutoff:val_cutoff].reset_index(drop=True)
    
    scaler_static = StandardScaler()
    X_S_train_sc = scaler_static.fit_transform(X_S_train)
    X_S_val_sc = scaler_static.transform(X_S_val)
    
    # --- 4. ARQUITECTURA SIAMESA ---
    tf.get_logger().setLevel('ERROR')
    
    input_H = Input(shape=(opt_timesteps, len(seq_features)))
    lstm_H = LSTM(opt_lstm_units, activation='tanh')(input_H)
    lstm_H = BatchNormalization()(lstm_H)
    lstm_H = Dropout(opt_dropout)(lstm_H)
    
    input_A = Input(shape=(opt_timesteps, len(seq_features)))
    lstm_A = LSTM(opt_lstm_units, activation='tanh')(input_A)
    lstm_A = BatchNormalization()(lstm_A)
    lstm_A = Dropout(opt_dropout)(lstm_A)
    
    input_S = Input(shape=(X_static.shape[1],))
    dense_S = Dense(16, activation='relu')(input_S)
    
    concat = Concatenate()([lstm_H, lstm_A, dense_S])
    x = Dense(opt_dense_units, activation='relu')(concat)
    x = BatchNormalization()(x)
    x = Dropout(opt_dropout)(x)
    output = Dense(3, activation='softmax')(x)
    
    modelo_lstm = Model(inputs=[input_H, input_A, input_S], outputs=output)
    modelo_lstm.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=opt_lr), loss='sparse_categorical_crossentropy')
    
    es = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True, verbose=0)
    
    modelo_lstm.fit([X_H_train, X_A_train, X_S_train_sc], y_train, validation_split=0.2, 
                    epochs=20, batch_size=64, callbacks=[es], verbose=0)
    
    # --- 5. EVALUACIÓN EN VALIDACIÓN ---
    p_lstm = modelo_lstm.predict([X_H_val, X_A_val, X_S_val_sc], verbose=0)
    
    saldo = 0.0
    apostado = 0
    
    for i in range(len(df_val_real)):
        row = df_val_real.iloc[i]
        odds = [row['Odds_Home'], row['Odds_Draw'], row['Odds_Away']]
        probs = p_lstm[i]
        real_target = y_val[i] 
        
        odd_1X = (odds[0] * odds[1]) / (odds[0] + odds[1])
        odd_X2 = (odds[1] * odds[2]) / (odds[1] + odds[2])
        ev_1 = (probs[0] * odds[0]) - 1; ev_2 = (probs[2] * odds[2]) - 1
        ev_1X = ((probs[0] + probs[1]) * odd_1X) - 1; ev_X2 = ((probs[1] + probs[2]) * odd_X2) - 1
        
        bet = None
        if ev_1 > opt_umbral_ev and probs[0] > 0.45: bet = {'res': [0], 'odd': odds[0]}
        elif ev_1X > opt_umbral_ev: bet = {'res': [0, 1], 'odd': odd_1X}
        elif ev_2 > opt_umbral_ev and probs[2] > 0.45: bet = {'res': [2], 'odd': odds[2]}
        elif ev_X2 > opt_umbral_ev: bet = {'res': [1, 2], 'odd': odd_X2}
        
        if bet:
            apostado += 10
            if real_target in bet['res']: saldo += (10 * bet['odd']) - 10
            else: saldo -= 10
            
    if apostado < 100: return -100.0 
    return (saldo / apostado) * 100

optuna.logging.set_verbosity(optuna.logging.WARNING)
study_lstm = optuna.create_study(direction='maximize')
study_lstm.optimize(objetivo_lstm_honesto, n_trials=10) # Puedes subir n_trials a 15 o 20 si tienes tiempo

print("\n🏆 ¡BÚSQUEDA OPTUNA TERMINADA!")
best_params = study_lstm.best_params
print(f"💰 MEJOR ROI en Validación: {study_lstm.best_value:.2f}%")
print(f"👉 Timesteps Elegidos (Memoria): {best_params['timesteps']} partidos")

# =======================================================
# 🚀 FASE FINAL: ENTRENAMIENTO Y TEST EN LA BÓVEDA CIEGA
# =======================================================


🏆 ¡BÚSQUEDA OPTUNA TERMINADA!
💰 MEJOR ROI en Validación: 11.42%
👉 Timesteps Elegidos (Memoria): 3 partidos


In [8]:
# Reconstruimos los tensores una última vez con el timestep ganador
X_seq_home_f, X_seq_away_f, X_static_f, y_target_f, indices_f = [], [], [], [], []
opt_timesteps_f = best_params['timesteps']

for i in range(len(df_base)):
    row = df_base.iloc[i]
    fecha, local, visita = row['Date'], row['HomeTeam'], row['AwayTeam']
    
    hist_L = dict_historial[local]
    hist_L = hist_L[hist_L['Date'] < fecha].tail(opt_timesteps_f)
    
    hist_V = dict_historial[visita]
    hist_V = hist_V[hist_V['Date'] < fecha].tail(opt_timesteps_f)
    
    if len(hist_L) == opt_timesteps_f and len(hist_V) == opt_timesteps_f:
        X_seq_home_f.append(hist_L[seq_features].values)
        X_seq_away_f.append(hist_V[seq_features].values)
        
        elo_diff = (row['Elo_Home'] + 50) - row['Elo_Away']
        X_static_f.append([elo_diff, row['Odds_Home'], row['Odds_Draw'], row['Odds_Away']])
        
        if row['FTHG'] > row['FTAG']: target = 0
        elif row['FTHG'] == row['FTAG']: target = 1
        else: target = 2
        
        y_target_f.append(target)
        indices_f.append(i)

X_seq_home_f = np.array(X_seq_home_f)
X_seq_away_f = np.array(X_seq_away_f)
X_static_f = np.array(X_static_f)
y_target_f = np.array(y_target_f)
df_validos_f = df_base.iloc[indices_f].reset_index(drop=True)

# Corte Final: Train agrupa (0% al 85%) y probamos en el Blind Test (85% al 100%)
N_tensors_f = len(X_seq_home_f)
val_cutoff_f = int(N_tensors_f * 0.85)

X_H_train_super = X_seq_home_f[:val_cutoff_f]
X_H_test_blind = X_seq_home_f[val_cutoff_f:]

X_A_train_super = X_seq_away_f[:val_cutoff_f]
X_A_test_blind = X_seq_away_f[val_cutoff_f:]

X_S_train_super = X_static_f[:val_cutoff_f]
X_S_test_blind = X_static_f[val_cutoff_f:]

y_train_super = y_target_f[:val_cutoff_f]
y_test_blind = y_target_f[val_cutoff_f:]

df_test_blind = df_validos_f.iloc[val_cutoff_f:].reset_index(drop=True)

scaler_final = StandardScaler()
X_S_train_super_sc = scaler_final.fit_transform(X_S_train_super)
X_S_test_blind_sc = scaler_final.transform(X_S_test_blind)

# Armamos el modelo definitivo
input_H = Input(shape=(opt_timesteps_f, len(seq_features)))
lstm_H = LSTM(best_params['lstm_units'], activation='tanh')(input_H)
lstm_H = BatchNormalization()(lstm_H)
lstm_H = Dropout(best_params['dropout'])(lstm_H)

input_A = Input(shape=(opt_timesteps_f, len(seq_features)))
lstm_A = LSTM(best_params['lstm_units'], activation='tanh')(input_A)
lstm_A = BatchNormalization()(lstm_A)
lstm_A = Dropout(best_params['dropout'])(lstm_A)

input_S = Input(shape=(X_static_f.shape[1],))
dense_S = Dense(16, activation='relu')(input_S)

concat = Concatenate()([lstm_H, lstm_A, dense_S])
x = Dense(best_params['dense_units'], activation='relu')(concat)
x = BatchNormalization()(x)
x = Dropout(best_params['dropout'])(x)
output = Dense(3, activation='softmax')(x)

lstm_final = Model(inputs=[input_H, input_A, input_S], outputs=output)
lstm_final.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=best_params['lr']), loss='sparse_categorical_crossentropy')
es_final = EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True, verbose=0)

lstm_final.fit([X_H_train_super, X_A_train_super, X_S_train_super_sc], y_train_super, validation_split=0.2, 
                epochs=40, batch_size=64, callbacks=[es_final], verbose=0)

p_lstm_blind = lstm_final.predict([X_H_test_blind, X_A_test_blind, X_S_test_blind_sc], verbose=0)

# Simulación Real
saldo_test = 0.0
apostado_test = 0
aciertos_test = 0

for i in range(len(df_test_blind)):
    row = df_test_blind.iloc[i]
    odds = [row['Odds_Home'], row['Odds_Draw'], row['Odds_Away']]
    probs = p_lstm_blind[i]
    real_target = y_test_blind[i]
    
    odd_1X = (odds[0] * odds[1]) / (odds[0] + odds[1])
    odd_X2 = (odds[1] * odds[2]) / (odds[1] + odds[2])
    ev_1 = (probs[0] * odds[0]) - 1; ev_2 = (probs[2] * odds[2]) - 1
    ev_1X = ((probs[0] + probs[1]) * odd_1X) - 1; ev_X2 = ((probs[1] + probs[2]) * odd_X2) - 1
    
    bet = None
    if ev_1 > best_params['umbral_ev'] and probs[0] > 0.45: bet = {'res': [0], 'odd': odds[0]}
    elif ev_1X > best_params['umbral_ev']: bet = {'res': [0, 1], 'odd': odd_1X}
    elif ev_2 > best_params['umbral_ev'] and probs[2] > 0.45: bet = {'res': [2], 'odd': odds[2]}
    elif ev_X2 > best_params['umbral_ev']: bet = {'res': [1, 2], 'odd': odd_X2}
    
    if bet:
        apostado_test += 10
        if real_target in bet['res']: 
            saldo_test += (10 * bet['odd']) - 10
            aciertos_test += 1
        else: 
            saldo_test -= 10

roi_blind = (saldo_test / apostado_test) * 100 if apostado_test > 0 else 0
hit_rate_blind = (aciertos_test / (apostado_test/10)) * 100 if apostado_test > 0 else 0

print(f"\n=======================================")
print(f"📊 RESULTADOS ABSOLUTOS (TEST CIEGO LSTM)")
print(f"📈 ROI FINAL REAL: {roi_blind:.2f}%")
print(f"🎯 Hit Rate:       {hit_rate_blind:.2f}%")
print(f"💶 Ganancia Neta:  {saldo_test:.2f}€")
print(f"📊 Total Apostado: {apostado_test:.2f}€")
print(f"=======================================")


📊 RESULTADOS ABSOLUTOS (TEST CIEGO LSTM)
📈 ROI FINAL REAL: 11.97%
🎯 Hit Rate:       71.84%
💶 Ganancia Neta:  208.31€
📊 Total Apostado: 1740.00€


In [10]:
# Guardar el DataFrame limpio y procesado en la carpeta correcta
df_crudo.to_csv('data/processed/df_crudo.csv', index=False)
print("✅ Datos procesados exportados exitosamente.")

✅ Datos procesados exportados exitosamente.
